In [3]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os
from pathlib import Path
import json

# Setting up augmentations that also modify bounding boxes
transform = A.Compose([
    A.HorizontalFlip(p=0.5),  # Horizontal flip with a 50% probability
    A.RandomRotate90(p=0.5),  # Random rotation by 90 degrees with a 50% probability
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.5),  # Shift, scale, and rotate
    A.RandomBrightnessContrast(p=0.5),  # Brightness and contrast adjustment with a 50% probability
    A.HueSaturationValue(p=0.5),  # Adjust hue, saturation, and brightness with a 50% probability
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))  # Important: specify label format and fields

# Path to the folder with images and labels
image_folder = Path(r'C:\Users\user\Documents\GitHub\robotdreams_homework\project\dataset\train\images')
label_folder = Path(r'C:\Users\user\Documents\GitHub\robotdreams_homework\project\dataset\train\labels')

# Path to save augmented images and labels
output_image_folder = Path(r'C:\Users\user\Documents\GitHub\robotdreams_homework\project\dataset\train\augmented_images')
output_label_folder = Path(r'C:\Users\user\Documents\GitHub\robotdreams_homework\project\dataset\train\augmented_labels')
output_image_folder.mkdir(parents=True, exist_ok=True)
output_label_folder.mkdir(parents=True, exist_ok=True)

# Apply augmentation to each image and its corresponding labels
for image_path in image_folder.glob('*.jpg'):
    label_path = label_folder / (image_path.stem + '.txt')
    
    # Load the image
    image = cv2.imread(str(image_path))
    
    # Load labels (YOLO format)
    with open(label_path, 'r') as f:
        labels = f.read().strip().splitlines()
        boxes = []
        class_labels = []
        for label in labels:
            class_label, x_center, y_center, width, height = map(float, label.split())
            boxes.append([x_center, y_center, width, height])
            class_labels.append(int(class_label))
    
    # Apply augmentations
    augmented = transform(image=image, bboxes=boxes, class_labels=class_labels)
    augmented_image = augmented['image']
    augmented_boxes = augmented['bboxes']
    augmented_labels = augmented['class_labels']
    
    # Save the new image
    output_image_path = output_image_folder / f'aug_{image_path.name}'
    cv2.imwrite(str(output_image_path), augmented_image)
    
    # Save the new labels (in YOLO format)
    output_label_path = output_label_folder / f'aug_{image_path.stem}.txt'
    with open(output_label_path, 'w') as f:
        for bbox, class_label in zip(augmented_boxes, augmented_labels):
            f.write(f"{class_label} " + " ".join(map(str, bbox)) + "\n")

print(f"Augmentation completed, augmented images and labels saved to {output_image_folder} and {output_label_folder}")


Augmentation completed, augmented images and labels saved to C:\Users\user\Documents\GitHub\robotdreams_homework\project\dataset\train\augmented_images and C:\Users\user\Documents\GitHub\robotdreams_homework\project\dataset\train\augmented_labels
